# vLLM serving — self-contained (Colab GPU)

No repo, no token. **Runtime → Change runtime type → GPU (T4)**, then run the
cells top to bottom. Proves: (1) vLLM serves a model over an OpenAI-compatible
API, and (2) a small inline RAG answer is generated by that vLLM-served model.

In [ ]:
!pip -q install vllm openai

In [ ]:
# Start the vLLM OpenAI-compatible server (background), wait until ready
import subprocess, time, urllib.request
MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
server = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL, '--port', '8000', '--max-model-len', '2048',
])
def ready():
    try:
        return urllib.request.urlopen('http://localhost:8000/v1/models', timeout=3).status == 200
    except Exception:
        return False
for _ in range(60):
    if ready():
        print('vLLM ready'); break
    time.sleep(5)
else:
    print('server not ready yet — wait and re-run this check cell')

In [ ]:
# 1) Proof vLLM serves the model over the OpenAI API
import openai
client = openai.OpenAI(base_url='http://localhost:8000/v1', api_key='x')
r = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':'What is the molecular weight of ethanol?'}],
    max_tokens=64, temperature=0)
print('SERVED BY vLLM:', r.model)
print(r.choices[0].message.content)

In [ ]:
# 2) Tiny RAG whose GENERATION runs on the vLLM-served model
PASSAGES = [
    ('bbb', 'A molecule tends to cross the blood-brain barrier when its topological polar surface area is below about 90 square angstroms, its molecular weight is under ~450 daltons, and its lipophilicity is moderate.'),
    ('tpsa', 'Lower topological polar surface area correlates with better passive membrane permeability.'),
    ('rag', 'Retrieval-augmented generation supplies retrieved documents as context so the generated answer is grounded in that evidence.'),
]
def retrieve(q, k=2):
    qs = set(q.lower().split())
    scored = sorted(PASSAGES, key=lambda p: len(qs & set(p[1].lower().split())), reverse=True)
    return scored[:k]

question = 'What makes a molecule able to cross the blood-brain barrier?'
ctx = '\n'.join(f'- {t}' for _, t in retrieve(question))
resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':'Answer using ONLY the context. Be concise.'},
        {'role':'user','content':f'Context:\n{ctx}\n\nQuestion: {question}'},
    ], max_tokens=120, temperature=0)
print('RAG answer (generated by vLLM-served model):')
print(resp.choices[0].message.content)
# server.terminate()  # run to stop the server